# Paper15 extension: annual models and RET picks

RF/XGBoost × rolling 5/10-year training, annual refits for 2024/2025/2026 and monthly predictions through April 2026. Exactly 12 fits; no tuning. The strict monthly close-return target, audited cohorts and RET stay unchanged. BK/XOM's post-2023 route limits remain explicit.

This notebook shows **prediction performance and selections only**. This model phase reports no portfolio returns; the subsequent verified [75-month realization is in notebook16](16_walkforward_extension_backtest.ipynb). The common daily comparison is February 2020–April 2026, 75 months, using preserved old picks and 28 added months.

[Method and frozen compute scope](../docs/WALKFORWARD_EXTENSION_MODELS_20261027.md). Run All reads progress or independently verified small JSON reports. It starts no model loading, fitting, source read or replay.


In [ ]:
from pathlib import Path
import json, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dmv2").is_dir())
sys.path.insert(0, str(ROOT / "src"))
local = ROOT / "configs/walkforward_extension_models.local.json"
settings = json.loads(local.read_text()) if local.exists() else {}
MODEL_EXTENSION_RUN = Path(settings.get("output_directory", ROOT / "outputs/walkforward_extension_models/20261027-v1"))
from dmv2.extension_model_reporting import read_extension_models
saved = None
try:
    saved = read_extension_models(MODEL_EXTENSION_RUN)
except FileNotFoundError:
    print("Private model-run evidence is unavailable. No compute started.")
else:
    print(saved["status"], "| fits:", saved["fits_completed"], "/", saved["maximum_fits"], "| new backtests:", saved["new_backtests"])
    print("Active seconds:", round(saved["active_seconds"], 1), "| peak worker GiB:", round(saved["peak_worker_rss_bytes"] / 2**30, 2))
    if saved.get("zero_fit_selection_recovery"):
        print("Original side-dtype check failure preserved; separate zero-fit selection recovery verified. No models retrained.")


## Prediction performance: January 2024–April 2026

Log loss, ten-class mode accuracy and Brier score. Primary metrics give each of the 28 months equal weight; pooled metrics are separate. Five/ten-year arms have different complete-history cohorts. These are class-prediction metrics, not trading returns.


In [ ]:
if saved and saved["status"] == "COMPLETE_VERIFIED_MODELS_AND_SELECTION":
    import pandas as pd
    comparison = pd.DataFrame(saved["metrics"])
    display(comparison.set_index(["arm", "reference"])[["months", "prediction_rows", "labelled_rows", "missing_labels", "mean_monthly_log_loss", "mean_monthly_accuracy", "mean_monthly_brier_sum10"]])
else:
    print("Verified prediction metrics are pending.")


## Annual diagnostics and RET selections

2024 and 2025 each cover 12 outcome months; 2026 covers January–April only. All old pre-2024 names, sides, weights and RET scores are checked unchanged. The added endpoint-gap counts describe selected missing monthly returns; they do not establish how many daily economic paths will fail. Daily quotes/actions and execution are checked in the separately versioned notebook16 realization.


In [ ]:
if saved and saved["status"] == "COMPLETE_VERIFIED_MODELS_AND_SELECTION":
    display(pd.DataFrame(saved["annual_metrics"]).set_index(["arm", "year", "reference"]))
    display(pd.DataFrame(saved["selection_coverage"]).set_index("arm"))
    for limitation in saved["limitations"]:
        print("•", limitation)
